# Attempting to find bivariate relationships using R

In [2]:
install.packages(c("tidyverse", "janitor", "skimr"))

Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done



In [3]:
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.1     ✔ stringr   1.5.2
✔ ggplot2   4.0.0     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [4]:
getwd()
df <- read_csv("../../data_pipeline/data/cleaned/premier_league_cleaned.csv")
head(df)


[1] "/home/mauposad/CS-Projects/data-mine-mposadas/models/EDA"

Rows: 20 Columns: 319
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
dbl (319): seasonGoals_overall, seasonConceded_overall, seasonGoalsTotal_ove...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


seasonGoals_overall,seasonConceded_overall,seasonGoalsTotal_overall,seasonGoalsMin_overall,seasonScoredMin_overall,seasonConcededMin_overall,seasonGoalDifference_overall,seasonWinsNum_overall,seasonLossesNum_overall,seasonCS_overall,⋯,additional_info.freekicks_total_over215_overall,additional_info.freekicks_total_over225_overall,additional_info.freekicks_total_over235_overall,additional_info.freekicks_total_over245_overall,additional_info.freekicks_total_over255_overall,additional_info.freekicks_total_over265_overall,additional_info.freekicks_total_over275_overall,additional_info.freekicks_total_over285_overall,additional_info.freekicks_total_over295_overall,additional_info.freekicks_total_over305_overall
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
69,34,103,33,49,101,35,20,4,13,⋯,59,55,48,41,41,34,31,24,21,17
64,65,129,27,54,53,-1,11,22,6,⋯,88,88,88,83,79,71,71,50,28,28
72,44,116,30,48,78,28,21,9,13,⋯,54,46,42,25,17,13,13,13,8,8
33,80,113,30,103,43,-47,6,25,3,⋯,71,50,50,36,28,21,14,14,14,7
51,51,102,34,67,67,0,13,11,11,⋯,60,60,53,47,27,20,20,13,7,7
42,44,86,40,81,78,-2,11,12,12,⋯,79,64,64,43,36,28,28,21,21,14


In [5]:
cor_matrix <- cor(df, use = "complete.obs", method = "pearson")
dim(cor_matrix)
cor_matrix[1:5, 1:5]

[1] 319 319

,seasonGoals_overall,seasonConceded_overall,seasonGoalsTotal_overall,seasonGoalsMin_overall,seasonScoredMin_overall
seasonGoals_overall,1.0000000,-0.7236396,0.3943299,-0.3804129,-0.9388466
seasonConceded_overall,-0.7236396,1.0000000,0.3488995,-0.3525198,0.7754700
seasonGoalsTotal_overall,0.3943299,0.3488995,1.0000000,-0.9859232,-0.2422805
seasonGoalsMin_overall,-0.3804129,-0.3525198,-0.9859232,1.0000000,0.2327072
seasonScoredMin_overall,-0.9388466,0.7754700,-0.2422805,0.2327072,1.0000000


In [6]:
n <-nrow(df)
r <-cor_matrix
t_mat <- r* sqrt(n-2) / sqrt(1-r^2)
t_mat[1:5, 1:5]
# clean up steps to convert Infinity values to missing numeric values
diag(t_mat) <- NA_real_ 
t_mat[is.infinite(t_mat)] <- sign(t_mat[is.infinite(t_mat)]) * Inf


,seasonGoals_overall,seasonConceded_overall,seasonGoalsTotal_overall,seasonGoalsMin_overall,seasonScoredMin_overall
seasonGoals_overall,Inf,-4.448334,1.820519,-1.745162,-11.567746
seasonConceded_overall,-4.448334,Inf,1.579511,-1.598213,5.210837
seasonGoalsTotal_overall,1.820519,1.579511,Inf,-25.017627,-1.059475
seasonGoalsMin_overall,-1.745162,-1.598213,-25.017627,Inf,1.015162
seasonScoredMin_overall,-11.567746,5.210837,-1.059475,1.015162,Inf


## Debugging steps before comparing to our critical t-values

In [9]:
sum(is.na(t_mat))
sum(is.infinite(t_mat))
range(t_mat, na.rm = TRUE)
t_crit <- qt(0.975, df = n - 2)
t_crit
r_example <- cor_matrix[1,2]
t_manual <- r_example * sqrt(n - 2) / sqrt(1 - r_example^2)
t_manual
abs(t_manual) > t_crit
table(sig_mask)

[1] 319

[1] 0

[1] -25.01763 235.98956

[1] 2.100922

[1] -4.448334

[1] TRUE

sig_mask
FALSE  TRUE 
82902 18540 

In [8]:
# Number of observations
n <- nrow(df)

# Critical t-value for a 95% confidence level (two-tailed)
t_crit <- qt(0.975, df = n - 2)

# Compute absolute t-scores
t_abs <- abs(t_mat)

# Create a logical mask: TRUE if |t| > t_crit (significant)
sig_mask <- t_abs > t_crit

# Optional: check proportion or sample
mean(sig_mask)           # fraction of significant correlations
sig_mask[1:5, 1:5]       # preview top-left corner


[1] NA

,seasonGoals_overall,seasonConceded_overall,seasonGoalsTotal_overall,seasonGoalsMin_overall,seasonScoredMin_overall
seasonGoals_overall,NA,TRUE,FALSE,FALSE,TRUE
seasonConceded_overall,TRUE,NA,FALSE,FALSE,TRUE
seasonGoalsTotal_overall,FALSE,FALSE,NA,TRUE,FALSE
seasonGoalsMin_overall,FALSE,FALSE,TRUE,NA,FALSE
seasonScoredMin_overall,TRUE,TRUE,FALSE,FALSE,NA


In [10]:
# Apply significance mask: keep only significant correlations
sig_cor_matrix <- cor_matrix

# Replace non-significant correlations with NA
sig_cor_matrix[!sig_mask] <- NA_real_

# Optional checks
dim(sig_cor_matrix)      # should still be 319 x 319
sum(!is.na(sig_cor_matrix))  # count of significant correlations
sig_cor_matrix[1:5, 1:5] # preview top-left corner


[1] 319 319

[1] 18859

,seasonGoals_overall,seasonConceded_overall,seasonGoalsTotal_overall,seasonGoalsMin_overall,seasonScoredMin_overall
seasonGoals_overall,1.0000000,-0.7236396,NA,NA,-0.9388466
seasonConceded_overall,-0.7236396,1.0000000,NA,NA,0.7754700
seasonGoalsTotal_overall,NA,NA,1.0000000,-0.9859232,NA
seasonGoalsMin_overall,NA,NA,-0.9859232,1.0000000,NA
seasonScoredMin_overall,-0.9388466,0.7754700,NA,NA,1.0000000


In [12]:
# Convert matrix to long-form data frame
library(reshape2)
sig_long <- melt(sig_cor_matrix, varnames = c("Var1", "Var2"), value.name = "r")

# Remove NAs and duplicates (keep upper triangle)
sig_long <- subset(sig_long, !is.na(r) & Var1 != Var2)
sig_long <- sig_long[!duplicated(t(apply(sig_long[,1:2], 1, sort))), ]

# Sort by absolute correlation
sig_long <- sig_long[order(abs(sig_long$r), decreasing = TRUE), ]

# Show top relationships
head(sig_long, 10)


,Var1,Var2,r
,<fct>,<fct>,<dbl>
2250,winPercentage_overall,seasonWinsNum_overall,0.9998384
41282,wins_2hg_percentage_overall,wins_2hg_overall,0.9995397
2249,seasonPPG_overall,seasonWinsNum_overall,0.9885186
4802,winPercentage_overall,seasonPPG_overall,0.9880772
40963,wins_2hg_percentage_overall,points_2hg_overall,0.9873639
642,seasonGoalsMin_overall,seasonGoalsTotal_overall,-0.9859232
40962,wins_2hg_overall,points_2hg_overall,0.9858213
71683,additional_info.shot_conversion_rate_overall,additional_info.shots_per_goals_scored_overall,-0.9834751
51844,additional_info.2h_cards_for_avg_overall,additional_info.2h_cards_for_overall,0.9824131


In [16]:
# Count how many significant relationships we have
n_sig <- nrow(sig_long)
cat("Number of significant relationships:", n_sig, "\n")
 # 9270 significant relationships. 
# Quick structure check
str(sig_long)

# Preview first 10 rows (for readability)
head(sig_long, 30)

# Optional: print all (will be long!)
# View(sig_long)       # opens in RStudio viewer
# print(sig_long)      # prints to console (be careful, may be thousands of rows)


Number of significant relationships: 9270 
'data.frame':	9270 obs. of  3 variables:
 $ Var1: Factor w/ 319 levels "seasonGoals_overall",..: 17 131 16 17 131 4 130 227 166 98 ...
 $ Var2: Factor w/ 319 levels "seasonGoals_overall",..: 8 130 8 16 129 3 129 225 163 95 ...
 $ r   : num  1 1 0.989 0.988 0.987 ...


,Var1,Var2,r
,<fct>,<fct>,<dbl>
2250,winPercentage_overall,seasonWinsNum_overall,0.9998384
41282,wins_2hg_percentage_overall,wins_2hg_overall,0.9995397
2249,seasonPPG_overall,seasonWinsNum_overall,0.9885186
4802,winPercentage_overall,seasonPPG_overall,0.9880772
40963,wins_2hg_percentage_overall,points_2hg_overall,0.9873639
642,seasonGoalsMin_overall,seasonGoalsTotal_overall,-0.9859232
40962,wins_2hg_overall,points_2hg_overall,0.9858213
71683,additional_info.shot_conversion_rate_overall,additional_info.shots_per_goals_scored_overall,-0.9834751
51844,additional_info.2h_cards_for_avg_overall,additional_info.2h_cards_for_overall,0.9824131


# Thoughts
Dimensionality is much too high to find useful correlation between variables. Now I would need to switch my focus to correlation that leads to key stats that directly contribute to winning. Is there a way we can use what we have to generate useful insights for our "winning formula"? 
Currently doing research for finding a "winning formula" 
[https://www.researchgate.net/publication/395831694_Which_soccer_metrics_best_predict_winning_A_data-driven_analysis_across_Europe%27s_top_five_leagues]



1. Shots & shooting quality: total shots, shots on target, shooting accuracy, and average shot distance — capturing offensive effectiveness.
2. Possession & passing: total passes, successful passes, pass accuracy %, and possession % — measuring control and buildup play.
3. Defensive actions: tackles, clearances, saves, interceptions — showing how teams protect their goal.
4. Set-piece & discipline: corners, fouls, yellow/red cards — indicators of aggression and risk.